In [1]:
import sys
import os
sys.path.append("../")
sys.path.append("../../")

In [2]:
from scale_rl.common.wandb_utils import *

In [3]:
from scale_rl.envs.mujoco import MUJOCO_ALL, MUJOCO_RANDOM_SCORE, MUJOCO_TD3_SCORE

/home/nas4_user/youngdolee/anaconda3/envs/scale_rl/lib/python3.9/site-packages/glfw/__init__.py:916: GLFWError: (65544) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)


In [4]:
from rliable import library as rly
from rliable import metrics as rly_metrics
from rliable import plot_utils as rly_plot_utils

aggregate_func = lambda x: np.array([
  rly_metrics.aggregate_mean(x),
  rly_metrics.aggregate_median(x),
  rly_metrics.aggregate_iqm(x)])

In [5]:
eval_df = pd.read_csv('../results/hypersimba/hypersimba.csv', index_col=0)
eval_df = eval_df[eval_df["metric"] == "avg_return"]
eval_df = eval_df[eval_df["env_name"].isin(MUJOCO_ALL)]

In [6]:
MUJ_STEPS = 1e6 # 1M

In [7]:
_eval_df = eval_df
overall_df = []
for env_name in sorted(MUJOCO_ALL):
    env_df = _eval_df[_eval_df["env_name"] == env_name]
    assert max(env_df["env_step"]) == MUJ_STEPS
    env_df = env_df[env_df["env_step"] == MUJ_STEPS]
    num_seeds = env_df["seed"].nunique()

    grouped_data = env_df.groupby("env_step")["value"]

    env_steps = grouped_data.mean().index.values
    mean = float(grouped_data.mean().values)
    std_err = float(grouped_data.sem().values)

    print(f"{env_name} - number of seeds: {num_seeds}")
    print(f"{round(mean)} [{round(mean) - round(1.96 * std_err)}, {round(mean) + round(1.96 * std_err)}]")
    
    overall_df.append(env_df)
    
overall_df = pd.concat(overall_df)
mean = overall_df["value"].mean()
std_err = overall_df["value"].sem()
print(f"Overall: {round(mean, 3)} ± {round(1.96 * std_err, 3)}")

Ant-v4 - number of seeds: 10
7429 [7209, 7649]
HalfCheetah-v4 - number of seeds: 10
12022 [11640, 12404]
Hopper-v4 - number of seeds: 10
4054 [3929, 4179]
Humanoid-v4 - number of seeds: 10
10546 [10195, 10897]
Walker2d-v4 - number of seeds: 10
6938 [6691, 7185]
Overall: 8197.893 ± 796.06


In [8]:
print(eval_df["env_name"].unique())

['Humanoid-v4' 'Ant-v4' 'Walker2d-v4' 'Hopper-v4' 'HalfCheetah-v4']


In [9]:
_eval_df = normalize_score_with_random_and_base_score(eval_df, MUJOCO_RANDOM_SCORE, MUJOCO_TD3_SCORE)
overall_df = []
for env_name in MUJOCO_ALL:
    # if env_name == "Humanoid-v4":
    #     continue
    env_df = _eval_df[_eval_df["env_name"] == env_name]
    assert max(env_df["env_step"]) == MUJ_STEPS
    env_df = env_df[env_df["env_step"] == MUJ_STEPS]
    num_seeds = env_df["seed"].nunique()

    grouped_data = env_df.groupby("env_step")["value"]

    env_steps = grouped_data.mean().index.values
    mean = float(grouped_data.mean().values)
    std_err = float(grouped_data.sem().values)

    print(f"{env_name} - number of seeds: {num_seeds}")
    print(f"{round(mean, 3)} ± {round(1.96 * std_err, 3)}")
    
    overall_df.append(env_df)
    
overall_df = pd.concat(overall_df)
mean = overall_df["value"].mean()
std_err = overall_df["value"].sem()
print(f"Overall: {round(mean, 3)} ± {round(1.96 * std_err, 3)}")

HalfCheetah-v4 - number of seeds: 10
1.133 ± 0.035
Hopper-v4 - number of seeds: 10
1.258 ± 0.039
Walker2d-v4 - number of seeds: 10
1.759 ± 0.063
Ant-v4 - number of seeds: 10
1.869 ± 0.055
Humanoid-v4 - number of seeds: 10
2.067 ± 0.07
Overall: 1.617 ± 0.103


In [10]:
_eval_df = normalize_score_with_random_and_base_score(eval_df, MUJOCO_RANDOM_SCORE, MUJOCO_TD3_SCORE)
metric_matrix_dict = generate_metric_matrix_dict(
    _eval_df, 
    env_step=MUJ_STEPS, 
    metric_type="avg_return",
)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    metric_matrix_dict, aggregate_func, reps=10000
)

In [11]:
_aggregate_scores = aggregate_scores["hypersimba"]
_aggregate_score_cis = aggregate_score_cis["hypersimba"]
print(_aggregate_score_cis[0])
for i, score in enumerate(["Mean", "Median", "IQM"]):
    print(f"{score}: {round(_aggregate_scores[i], 2)} [{round(_aggregate_score_cis[0][i], 2)}, {round(_aggregate_score_cis[1][i], 2)}]")

[1.51537376 1.49569398 1.47330346]
Mean: 1.62 [1.52, 1.72]
Median: 1.62 [1.5, 1.75]
IQM: 1.64 [1.47, 1.79]
